## Export Variant Video Library

- pull these functions into the main library later

In [ ]:
import glob
import os
import pickle as pkl

import dask.array as da
import dask.dataframe as dd
import holoviews as hv
import numpy as np
import pandas as pd
import scipy as sp
import scipy.stats
import seaborn as sns
import sklearn as skl
from dask.distributed import wait
from matplotlib import pyplot as plt
from sklearn.linear_model import LinearRegression

import paulssonlab.deaton.trenchripper.trenchripper as tr

hv.extension("bokeh")

In [ ]:
# addition of active memory manager
import dask

dask.config.set({"distributed.scheduler.active-memory-manager.start": True})
dask.config.set({"distributed.scheduler.worker-ttl": "5m"})
dask.config.set({"distributed.scheduler.allowed-failures": 100})

dask_wd = "/home/lag36/scratch/lag36/dask"

In [ ]:
import copy

import cv2
import h5py
import skimage as sk
from skimage.color import gray2rgb


def lookups_to_output_arr(headpath, channel, t_range, x):
    for i in range(len(x)):
        with h5py.File(
            headpath + "/kymograph/kymograph_" + str(x.iloc[i]["File Index"]) + ".hdf5",
            "r",
        ) as infile:
            if i == 0:
                output_arr_list = []
            if x.iloc[i]["lane orientation"] == "top":
                # output_arr_dict[channel].append(infile[channel][x.iloc[i]["File Trench Index"]][:,::-1])
                output_arr_list.append(
                    infile[channel][x.iloc[i]["File Trench Index"]][
                        t_range[0] : t_range[1], ::-1
                    ]
                )
            else:
                # output_arr_dict[channel].append(infile[channel][x.iloc[i]["File Trench Index"]])
                output_arr_list.append(
                    infile[channel][x.iloc[i]["File Trench Index"]][
                        t_range[0] : t_range[1]
                    ]
                )
    output_arr = np.concatenate(output_arr_list, axis=2)
    return output_arr


def export_avi(out_handle, output_arr, min_intensity=0, max_intensity=10000, fps=30):
    output_arr_scaled = sk.exposure.rescale_intensity(
        output_arr, in_range=(min_intensity, max_intensity)
    )
    for t in range(output_arr_scaled.shape[0]):
        out_handle.write(sk.color.gray2rgb(sk.util.img_as_ubyte(output_arr_scaled[t])))


def lookup_to_avi(headpath_list, outpath, channel_params_dict, x, t_chunk_size=30):
    tr.writedir(outpath, overwrite=False)

    with open(headpath_list[0] + "/kymograph/metadata.pkl", "rb") as metafile:
        kymo_meta = pkl.load(metafile)
    y_dim, x_dim = (
        kymo_meta["kymograph_params"]["ttl_len_y"],
        kymo_meta["kymograph_params"]["trench_width_x"],
    )
    x_dim = (x_dim + 1) * len(x)
    global_meta = (
        tr.pandas_hdf5_handler(headpath_list[0] + "/metadata.hdf5")
        .read_df("global", read_metadata=True)
        .metadata
    )
    t_dim = global_meta["num_frames"]

    for channel in channel_params_dict.keys():
        channel_params = channel_params_dict[channel]
        min_intensity, max_intensity, fps = channel_params
        tr.writedir(outpath + "/" + channel, overwrite=False)
        fourcc = cv2.VideoWriter_fourcc(*"H264")  # LUIS
        out_handle = cv2.VideoWriter(
            outpath + "/" + channel + "/" + str(x.iloc[0]["oDEPool7_id Temp"]) + ".mp4",
            fourcc,
            fps,
            (x_dim, y_dim),
        )

        for init_t in range(0, t_dim, t_chunk_size):
            fin_t = min(init_t + t_chunk_size, t_dim + 1)
            t_range = [init_t, fin_t]
            output_arr_list = []
            for i, headpath in enumerate(headpath_list):
                filtered_x = x[x["Experiment #"] == i]
                if len(filtered_x) > 0:
                    output_arr = lookups_to_output_arr(
                        headpath, channel, t_range, filtered_x
                    )
                    output_arr_list.append(output_arr)
            merged_output_arr = np.concatenate(output_arr_list, axis=2)
            del output_arr_list

            export_avi(
                out_handle,
                merged_output_arr,
                min_intensity=min_intensity,
                max_intensity=max_intensity,
                fps=fps,
            )  # LUIS

        out_handle.release()

    with open(
        outpath + "/Lockfile." + str(x.iloc[0]["oDEPool7_id Temp"]) + ".txt", "w"
    ) as outfile:
        outfile.write("Done")

    return True

In [ ]:
dask_controller = tr.trcluster.dask_controller(
    walltime="2:00:00",
    local=False,
    n_workers=100,
    n_workers_min=100,
    memory="16GB",
    working_directory=dask_wd,
)
dask_controller.startdask()

In [ ]:
dask_controller.shutdown()

In [ ]:
dask_controller.daskclient

In [ ]:
def lookup_to_avi_all(
    output_phenotype_kmyo_df_path_list,
    headpath_list,
    output_path,
    channel_params_dict,
    n_partitions=1000,
    overwrite=False,
):
    output_phenotype_kmyo_df_list = [
        dd.read_parquet(item, engine="pyarrow", calculate_divisions=True)
        for item in output_phenotype_kmyo_df_path_list
    ]

    variant_to_trench_df_list = []

    for i in range(len(output_phenotype_kmyo_df_list)):
        trench_to_lookup_df = (
            output_phenotype_kmyo_df_list[i]
            .set_index("phenotype trenchid", sorted=True)
            .groupby("phenotype trenchid")
            .apply(lambda x: x.iloc[0])
        )
        trench_to_lookup_df = trench_to_lookup_df[
            ["File Index", "File Trench Index", "lane orientation", "oDEPool7_id"]
        ].compute()
        trench_to_lookup_df["oDEPool7_id Temp"] = copy.deepcopy(
            trench_to_lookup_df["oDEPool7_id"]
        )
        trench_to_lookup_df["Experiment #"] = i
        variant_to_trench_df = trench_to_lookup_df.reset_index().set_index(
            "oDEPool7_id"
        )
        variant_to_trench_df_list.append(variant_to_trench_df)

    variant_to_trench_df = pd.concat(variant_to_trench_df_list, axis=0).sort_index()
    variant_to_trench_dd = dd.from_pandas(
        variant_to_trench_df, npartitions=n_partitions
    ).persist()

    if overwrite:
        finished_files = []
        print("Starting Run.")
    else:
        # Check for existing files in case of in progress run
        if os.path.exists(output_path):
            if os.path.exists(output_path + "/Finished_Run.txt"):
                print("Run Already Finished.")
                return True
            else:
                output_path_dirlist = os.listdir(output_path)
                finished_files = [
                    int(path.split(".")[1])
                    for path in output_path_dirlist
                    if "Lockfile" in path
                ]
                print("Resuming Run.")
        else:
            finished_files = []
            print("No Previous Run. Starting Run.")

    tr.writedir(output_path, overwrite=overwrite)

    variant_to_trench_dd = variant_to_trench_dd.reset_index().persist()
    wait(variant_to_trench_dd)
    variant_to_trench_dd = variant_to_trench_dd[
        ~variant_to_trench_dd["oDEPool7_id"].isin(finished_files)
    ].persist()
    wait(variant_to_trench_dd)
    variant_to_trench_dd = variant_to_trench_dd.set_index("oDEPool7_id", sorted=True)
    wait(variant_to_trench_dd)

    variant_to_trench_dd.groupby("oDEPool7_id").apply(
        lambda x: lookup_to_avi(headpath_list, output_path, channel_params_dict, x),
        meta=bool,
    ).compute()

    print("Cleaning Up...")

    for filename in glob.glob(output_path + "Lockfile*"):
        os.remove(filename)

    with open(output_path + "/Finished_Run.txt", "w") as outfile:
        outfile.write("Done.")

    print("Done.")
    return True

In [ ]:
print(1)

In [ ]:
output_phenotype_kmyo_df_path_list = [
    "/home/lag36/scratch/lag36/2023-03-11_lDE26_Run_1/2023-04-21_lDE26_Run_1_Final_Barcodes_df/"
]

headpath_list = ["/home/lag36/scratch/lag36/2023-03-11_lDE26_Run_1/Growth_Division"]

output_path = "/home/lag36/scratch/lag36/2023-03-11_lDE26_Run_1/MP4_Library/"  # Last / is key here. Otherwise it will crash
channel_params_dict = {"YFP": (0, 30000, 10), "mCherry": (0, 20000, 10)}

lookup_to_avi_all(
    output_phenotype_kmyo_df_path_list,
    headpath_list,
    output_path,
    channel_params_dict,
    overwrite=True,
)

## Merging different channels into a single video (Luis)
This tool merges exported videos from `4_Video_Export.ipynb` into a single video of all imaging channels combined. It used `videos_grid.py`, which should be included in the same directory

In [ ]:
import os

import videos_grid
from IPython.display import Video

Please set the following paths and channels. Videos should be stored in `headpath_videos/channels[i]` for all channels in `channels`

In [ ]:
headpath_videos = "/home/lag36/scratch/lag36/2023-03-11_lDE26_Run_1/MP4_Library"  # No /
output_path = (
    "/home/lag36/scratch/lag36/2023-03-11_lDE26_Run_1/MP4_Library/Merged/"  # With /
)
channels = ["mCherry", "YFP"]

mch_files = os.listdir(headpath_videos + "/" + channels[0])
yfp_files = os.listdir(headpath_videos + "/" + channels[1])

mch_ids = sorted([file.split(".")[0] for file in mch_files])
yfp_ids = sorted([file.split(".")[0] for file in yfp_files])

mch_ids == yfp_ids

In [ ]:
# Function to merge videos
def merge_video(
    oDEPool7_id,
    headpath_videos=headpath_videos,
    output_path=output_path,
    channels=["mCherry", "YFP"],
):
    videos_to_merge = [
        headpath_videos + "/" + channel + "/" + oDEPool7_id + ".mp4"
        for channel in channels
    ]
    videos_grid.merge_videos(
        videos_to_merge,
        output_path + oDEPool7_id + ".mp4",
        grid_size=(2, 1),  # 2 row, 1 cols
        max_frames=61,
    )
    # merge first 100 frames per video
    return oDEPool7_id

In [ ]:
# Run for entire dataset
tr.writedir(output_path + "/", overwrite=True)
futures_dict = {}
for oDEPool7_id in mch_ids:
    futures_dict[
        "oDEPool7_id: " + str(oDEPool7_id)
    ] = dask_controller.daskclient.submit(merge_video, oDEPool7_id)

#### Debug for a single frame (lysS)

In [ ]:
# oDEPool7_id = '24808'
# videos_to_merge = [headpath_videos + '/' + channel + '/' + oDEPool7_id + '.mp4' for channel in channels]
# output_full_path = output_path+ oDEPool7_id+'.mp4'

# videos_grid.merge_videos(videos_to_merge,
#                          output_full_path,
#                          grid_size=(2, 1),  # 2 row, 1 cols
#                          max_frames=61)

## Load dataframe for gene to oDEPool7_id lookup

In [ ]:
import dask.dataframe as dd

In [ ]:
df_bc = dd.read_parquet(
    "/home/lag36/scratch/lag36/2023-03-11_lDE26_Run_1/2023-04-21_lDE26_Run_1_Final_Barcodes_df",
    calculate_divisions=True,
)
df_id = df_bc.groupby(["oDEPool7_id"]).first().compute()
df_id["Gene"].to_csv("./df_id_lDE26.csv")

In [ ]:
df = pd.read_csv("./df_id_lDE26.csv")

In [ ]:
df.dropna()[df.dropna()["Gene"].str.contains("rplL")]

In [ ]:
dask_controller.shutdown()

In [ ]:
# rpsR,

In [ ]:
from IPython.display import Video

oDE_id = 704
Video(
    "/home/lag36/scratch/lag36/2023-03-11_lDE26_Run_1/MP4_Library/Merged/"
    + str(oDE_id)
    + ".mp4",
    embed=True,
)

In [ ]:
from IPython.display import Video

oDE_id = 704
Video(
    "/home/lag36/scratch/lag36/2023-03-25_lDE28_Run_1/MP4_Library/Merged/"
    + str(oDE_id)
    + ".mp4",
    embed=True,
)

In [ ]:
dask_controller.shutdown()

# Fluorescence segmentation video debugging

In [ ]:
def lookups_to_output_arr(headpath, channel, t_range, x):
    for i in range(len(x)):
        with h5py.File(
            headpath + "/kymograph/kymograph_" + str(x.iloc[i]["File Index"]) + ".hdf5",
            "r",
        ) as infile:
            if i == 0:
                output_arr_list = []
            if x.iloc[i]["lane orientation"] == "top":
                # output_arr_dict[channel].append(infile[channel][x.iloc[i]["File Trench Index"]][:,::-1])
                output_arr_list.append(
                    infile[channel][x.iloc[i]["File Trench Index"]][
                        t_range[0] : t_range[1], ::-1
                    ]
                )
            else:
                # output_arr_dict[channel].append(infile[channel][x.iloc[i]["File Trench Index"]])
                output_arr_list.append(
                    infile[channel][x.iloc[i]["File Trench Index"]][
                        t_range[0] : t_range[1]
                    ]
                )
    output_arr = np.concatenate(output_arr_list, axis=2)
    return output_arr


def export_avi(out_handle, output_arr, min_intensity=0, max_intensity=10000, fps=30):
    output_arr_scaled = sk.exposure.rescale_intensity(
        output_arr, in_range=(min_intensity, max_intensity)
    )
    for t in range(output_arr_scaled.shape[0]):
        out_handle.write(sk.color.gray2rgb(sk.util.img_as_ubyte(output_arr_scaled[t])))


def lookup_to_avi(headpath_list, outpath, channel_params_dict, x, t_chunk_size=30):
    tr.writedir(outpath, overwrite=False)

    with open(headpath_list[0] + "/kymograph/metadata.pkl", "rb") as metafile:
        kymo_meta = pkl.load(metafile)
    y_dim, x_dim = (
        kymo_meta["kymograph_params"]["ttl_len_y"],
        kymo_meta["kymograph_params"]["trench_width_x"],
    )
    x_dim = (x_dim + 1) * len(x)
    global_meta = (
        tr.pandas_hdf5_handler(headpath_list[0] + "/metadata.hdf5")
        .read_df("global", read_metadata=True)
        .metadata
    )
    t_dim = global_meta["num_frames"]

    for channel in channel_params_dict.keys():
        channel_params = channel_params_dict[channel]
        min_intensity, max_intensity, fps = channel_params
        tr.writedir(outpath + "/" + channel, overwrite=False)
        fourcc = cv2.VideoWriter_fourcc(*"H264")  # LUIS
        out_handle = cv2.VideoWriter(
            outpath + "/" + channel + "/" + str(x.iloc[0]["oDEPool7_id Temp"]) + ".mp4",
            fourcc,
            fps,
            (x_dim, y_dim),
        )

        for init_t in range(0, t_dim, t_chunk_size):
            fin_t = min(init_t + t_chunk_size, t_dim + 1)
            t_range = [init_t, fin_t]
            output_arr_list = []
            for i, headpath in enumerate(headpath_list):
                filtered_x = x[x["Experiment #"] == i]
                if len(filtered_x) > 0:
                    output_arr = lookups_to_output_arr(
                        headpath, channel, t_range, filtered_x
                    )
                    output_arr_list.append(output_arr)
            merged_output_arr = np.concatenate(output_arr_list, axis=2)
            del output_arr_list

            export_avi(
                out_handle,
                merged_output_arr,
                min_intensity=min_intensity,
                max_intensity=max_intensity,
                fps=fps,
            )  # LUIS

        out_handle.release()

    with open(
        outpath + "/Lockfile." + str(x.iloc[0]["oDEPool7_id Temp"]) + ".txt", "w"
    ) as outfile:
        outfile.write("Done")

    return True

In [ ]:
output_phenotype_kmyo_df_path_list = [
    "/home/lag36/scratch/lag36/2023-03-11_lDE26_Run_1/2023-04-21_lDE26_Run_1_Final_Barcodes_df"
]

headpath_list = ["/home/lag36/scratch/lag36/2023-03-11_lDE26_Run_1/Growth_Division"]

output_path = "/home/lag36/scratch/lag36/2023-03-11_lDE26_Run_1/MP4_Library/"  # Last / is key here. Otherwise it will crash
channel_params_dict = {"YFP": (0, 30000, 10), "mCherry": (0, 20000, 10)}

lookup_to_avi_all(
    output_phenotype_kmyo_df_path_list,
    headpath_list,
    output_path,
    channel_params_dict,
    overwrite=True,
)

In [ ]:
def lookup_to_avi_all(
    output_phenotype_kmyo_df_path_list,
    headpath_list,
    output_path,
    channel_params_dict,
    n_partitions=1000,
    overwrite=False,
):
    output_phenotype_kmyo_df_list = [
        dd.read_parquet(item, engine="pyarrow", calculate_divisions=True)
        for item in output_phenotype_kmyo_df_path_list
    ]

    variant_to_trench_df_list = []

    for i in range(len(output_phenotype_kmyo_df_list)):
        trench_to_lookup_df = (
            output_phenotype_kmyo_df_list[i]
            .set_index("phenotype trenchid", sorted=True)
            .groupby("phenotype trenchid")
            .apply(lambda x: x.iloc[0])
        )
        trench_to_lookup_df = trench_to_lookup_df[
            ["File Index", "File Trench Index", "lane orientation", "oDEPool7_id"]
        ].compute()
        trench_to_lookup_df["oDEPool7_id Temp"] = copy.deepcopy(
            trench_to_lookup_df["oDEPool7_id"]
        )
        trench_to_lookup_df["Experiment #"] = i
        variant_to_trench_df = trench_to_lookup_df.reset_index().set_index(
            "oDEPool7_id"
        )
        variant_to_trench_df_list.append(variant_to_trench_df)

    variant_to_trench_df = pd.concat(variant_to_trench_df_list, axis=0).sort_index()
    variant_to_trench_dd = dd.from_pandas(
        variant_to_trench_df, npartitions=n_partitions
    ).persist()

    if overwrite:
        finished_files = []
        print("Starting Run.")
    else:
        # Check for existing files in case of in progress run
        if os.path.exists(output_path):
            if os.path.exists(output_path + "/Finished_Run.txt"):
                print("Run Already Finished.")
                return True
            else:
                output_path_dirlist = os.listdir(output_path)
                finished_files = [
                    int(path.split(".")[1])
                    for path in output_path_dirlist
                    if "Lockfile" in path
                ]
                print("Resuming Run.")
        else:
            finished_files = []
            print("No Previous Run. Starting Run.")

    tr.writedir(output_path, overwrite=overwrite)

    variant_to_trench_dd = variant_to_trench_dd.reset_index().persist()
    wait(variant_to_trench_dd)
    variant_to_trench_dd = variant_to_trench_dd[
        ~variant_to_trench_dd["oDEPool7_id"].isin(finished_files)
    ].persist()
    wait(variant_to_trench_dd)
    variant_to_trench_dd = variant_to_trench_dd.set_index("oDEPool7_id", sorted=True)
    wait(variant_to_trench_dd)

    variant_to_trench_dd.groupby("oDEPool7_id").apply(
        lambda x: lookup_to_avi(headpath_list, output_path, channel_params_dict, x),
        meta=bool,
    ).compute()

    print("Cleaning Up...")

    for filename in glob.glob(output_path + "Lockfile*"):
        os.remove(filename)

    with open(output_path + "/Finished_Run.txt", "w") as outfile:
        outfile.write("Done.")

    print("Done.")
    return True

### Older Functions

### Interactive Variant Overlay

In [ ]:
# def linked_kymograph_for_gene_table(xrstack,wrapped_xrstack,df,select_gene,select_trenchid,select_unpacked_trenchid=None,trenchid_column='trenchid',
#                                     y_scale=3,x_window_size=300,rescale_videos=True,fps=10,dpi=200):
#     ### stream must return trenchid value
#     #### df must have trenchid lookups
#     width,height = xrstack.shape[3],int(xrstack.shape[2]*y_scale)
#     x_window_scale = x_window_size/xrstack.shape[3]
#     x_size = int(xrstack.shape[3]*(y_scale*x_window_scale))

#     df = df.reset_index(drop=False).set_index("Gene",drop=False).sort_index()
#     df.index = df.index.rename("Gene Index")
#     df_genes_only = df[["Gene"]].groupby("Gene").apply(lambda x: x.iloc[0])
#     dataset = hv.Dataset(df)
#     gene_dataset = hv.Dataset(df_genes_only)

#     if select_unpacked_trenchid != None:
#         def select_pt_load_image(channel,select_gene_index,trenchid_index,unpack_trenchid_index,width=width,height=height):
#             gene_name = gene_dataset.iloc[select_gene_index]["Gene"][0]
#             filtered_dataset = hv.Dataset(df.loc[gene_name:gene_name])
#             double_filtered_dataset = hv.Dataset({"trenchid": filtered_dataset.iloc[trenchid_index][trenchid_column][0]},['trenchid'])
#             trenchid = double_filtered_dataset.iloc[unpack_trenchid_index]["trenchid"][0]
#             arr = xrstack.loc[channel,trenchid].compute(scheduler="threads").values
#             return hv.Image(arr,bounds=(0,0,width,height))

#         def print_trenchid(select_gene_index,trenchid_index,unpack_trenchid_index):
#             gene_name = gene_dataset.iloc[select_gene_index]["Gene"][0]
#             filtered_dataset = hv.Dataset(df.loc[gene_name:gene_name])
#             double_filtered_dataset = hv.Dataset({"trenchid": filtered_dataset.iloc[trenchid_index][trenchid_column][0]},['trenchid'])
#             trenchid = double_filtered_dataset.iloc[unpack_trenchid_index]["trenchid"][0]
#             return hv.Text(3.,20.,str(trenchid),fontsize=30)

#     else:
#         def select_pt_load_image(channel,select_gene_index,trenchid_index,width=width,height=height):
#             gene_name = gene_dataset.iloc[select_gene_index]["Gene"][0]
#             filtered_dataset = hv.Dataset(df.loc[gene_name:gene_name])
#             trenchid = filtered_dataset.iloc[trenchid_index][trenchid_column][0]
#             arr = xrstack.loc[channel,trenchid].compute(scheduler="threads").values
#             return hv.Image(arr,bounds=(0,0,width,height))

#         def print_trenchid(select_gene_index,trenchid_index):
#             gene_name = gene_dataset.iloc[select_gene_index]["Gene"][0]
#             filtered_dataset = hv.Dataset(df.loc[gene_name:gene_name])
#             trenchid = filtered_dataset.iloc[trenchid_index][trenchid_column][0]
#             return hv.Text(3.,20.,str(trenchid),fontsize=30)

#     def set_bounds(fig, element, y_dim=height, x_dim=width, x_window_size=x_window_size):
#         sy = y_dim-0.5
#         sx = x_dim-0.5

#         x_zoom_pos = (fig.state.x_range.start,fig.state.x_range.end)
#         xdata = element.dataset.data['x']-0.5
#         xdata_range = (xdata.min(), xdata.max()+1.)
#         init_test = x_zoom_pos==xdata_range

#         fig.state.x_range.bounds = (0, sx)
#         fig.state.y_range.bounds = (-0.5, sy)

#         if init_test:
#             fig.state.x_range.start = 0
#             fig.state.x_range.end = x_window_size
#         else:
#             fig.state.x_range.start = x_zoom_pos[0]
#             fig.state.x_range.end = x_zoom_pos[0]+x_window_size

#         fig.state.y_range.start = 0
#         fig.state.y_range.end = y_dim
#         fig.state.x_range.reset_start = 0
#         fig.state.x_range.reset_end = x_window_size
#         fig.state.y_range.reset_start = 0
#         fig.state.y_range.reset_end = y_dim

#     if select_unpacked_trenchid != None:
#         image_stack = hv.DynamicMap(select_pt_load_image, kdims=['Channel'], streams=[select_gene,select_trenchid,select_unpacked_trenchid])
#         trenchid_display = hv.DynamicMap(print_trenchid, streams=[select_gene,select_trenchid,select_unpacked_trenchid])
#     else:
#         image_stack = hv.DynamicMap(select_pt_load_image, kdims=['Channel'], streams=[select_gene,select_trenchid])
#         trenchid_display = hv.DynamicMap(print_trenchid, streams=[select_gene,select_trenchid])

#     kymograph_display = image_stack.opts(plot={'Image': dict(colorbar=True, tools=['hover'],hooks=[set_bounds],aspect='equal'),})
#     kymograph_display = kymograph_display.opts(cmap='Greys_r',height=height,width=x_size)

#     kymograph_display = kymograph_display.redim.range(trenchid=(0,xrstack.shape[1]))
#     kymograph_display = kymograph_display.redim.values(Channel=xrstack.coords["Channel"].values.tolist())

#     if select_unpacked_trenchid != None:

#         def save_video(event):
#             hv.extension('matplotlib')
#             channel = kymograph_display.dimension_values("Channel")[0]
#             gene_name = gene_dataset.iloc[select_gene.index[0]]["Gene"][0]
#             filtered_dataset = hv.Dataset(df.loc[gene_name:gene_name])
#             double_filtered_dataset = hv.Dataset({"trenchid": filtered_dataset.iloc[select_trenchid.index[0]][trenchid_column][0]},['trenchid'])
#             trenchid = double_filtered_dataset.iloc[select_unpacked_trenchid.index[0]]["trenchid"][0]
#             arr = wrapped_xrstack.loc[channel,trenchid].values
#             if rescale_videos:
#                 arr = np.array([sk.exposure.rescale_intensity(arr[t]) for t in range(arr.shape[0])])
#             hmap = hv.HoloMap([(t, hv.Raster(arr[t])) for t in range(0,arr.shape[0])], kdims=['t'])
#             hmap = hmap.opts(plot={'Raster': dict(tools=['hover'],aspect='equal',cmap='Greys_r'),})
#             hmap = hmap.opts(cmap='Greys_r')
#             hv.save(hmap, gene_name + '_trenchid=' + str(trenchid) + '.mp4', fmt='mp4', fps=fps, dpi=dpi , backend='matplotlib')
#             hv.extension('bokeh')

#     else:

#         def save_video(event):
#             hv.extension('matplotlib')
#             channel = kymograph_display.dimension_values("Channel")[0]
#             gene_name = gene_dataset.iloc[select_gene.index[0]]["Gene"][0]
#             filtered_dataset = hv.Dataset(df.loc[gene_name:gene_name])
#             trenchid = filtered_dataset.iloc[select_trenchid.index[0]][trenchid_column][0]
#             arr = wrapped_xrstack.loc[channel,trenchid].values
#             if rescale_videos:
#                 arr = np.array([sk.exposure.rescale_intensity(arr[t]) for t in range(arr.shape[0])])
#             hmap = hv.HoloMap([(t, hv.Raster(arr[t])) for t in range(0,arr.shape[0])], kdims=['t'])
#             hmap = hmap.opts(plot={'Raster': dict(tools=['hover'],aspect='equal',cmap='Greys_r'),})
#             hmap = hmap.opts(cmap='Greys_r')
#             hv.save(hmap, gene_name + '_trenchid=' + str(trenchid) + '.mp4', fmt='mp4', fps=fps, dpi=dpi , backend='matplotlib')
#             hv.extension('bokeh')

#     trenchid_display = trenchid_display.opts(text_align="left",text_color="white")

#     output_display = kymograph_display*trenchid_display

#     button = pn.widgets.button.Button(name='Save Video', width=60)
#     button.on_click(save_video)

#     return output_display, button

In [ ]:
output_phenotype_kmyo_df = dd.read_parquet(
    "/home/de64/scratch/de64/sync_folder/2022-07-27_lDE20_Merged_Analysis/2022-07-31_lDE20_Final_Barcodes_df_Merged/",
    calculate_divisions=True,
)

In [ ]:
output_phenotype_kmyo_df.index

In [ ]:
headpath = "/home/de64/scratch/de64/sync_folder/2022-06-16_lDE23_Run_3/Growth_Division/"
# note: shutdown dask when doing this...fix bug later
overlay_handle = tr.variant_overlay(
    headpath,
    "/home/de64/scratch/de64/sync_folder/2022-06-16_lDE23_Run_3/2022-07-01_lDE23_Final_Barcodes_df/",
    display_values_list=["Gene", "TargetID", "N Mismatch"],
    persist_data=False,
)  ##fix this, was improperly made (only initial cellID timepoints)

In [ ]:
overlay_handle.view_overlay(vmin=0, vmax=10000)

### Interactive Variant Table

In [ ]:
gene_table = dd.read_parquet(
    "/home/de64/scratch/lag36/2022-08-19_MM-Ti5-lDE24/2022-09-08_lDE24_FInal_Barcodes_df_Run_1",
    engine="pyarrow",
    calculate_divisions=True,
)
gene_table = gene_table.reset_index().set_index("phenotype trenchid", sorted=True)
gene_table = (
    gene_table.groupby("phenotype trenchid", sort=False)
    .apply(lambda x: x.iloc[0])
    .reset_index()
    .set_index("FOV Parquet Index")
)
gene_table_out = gene_table.groupby("sgRNA").apply(lambda x: x.iloc[0])
gene_table_out["phenotype trenchids"] = gene_table.groupby("sgRNA").apply(
    lambda x: x["phenotype trenchid"].tolist()
)
gene_table_out = gene_table_out[
    [
        "Gene",
        "Target Sequence",
        "phenotype trenchids",
        "N Mismatch",
        "N Target Sites",
        "Category",
        "Strand",
    ]
].compute()

In [ ]:
kymo_xarr = tr.kymo_xarr(
    "/home/de64/scratch/lag36/2022-08-19_MM-Ti5-lDE24/Growth_Division"
)
wrapped_kymo_xarr = tr.kymo_xarr(
    "/home/de64/scratch/lag36/2022-08-19_MM-Ti5-lDE24/Growth_Division", unwrap=False
)

In [ ]:
(
    gene_table_layout,
    select_gene,
    select_trenchid,
    select_unpacked_trenchid,
) = tr.linked_gene_table(
    gene_table_out,
    index_key="Gene",
    trenchids_as_list=True,
    trenchid_column="phenotype trenchids",
)

In [ ]:
gene_table_layout

In [ ]:
output_display, save_button = tr.linked_kymograph_for_gene_table(
    kymo_xarr,
    wrapped_kymo_xarr,
    gene_table_out,
    select_gene,
    select_trenchid,
    select_unpacked_trenchid=select_unpacked_trenchid,
    trenchid_column="phenotype trenchids",
    y_scale=1.2,
    x_window_size=1200,
)

In [ ]:
output_display

In [ ]:
dask_controller.shutdown()